###IMPORTAÇÂO

In [1]:
import csv
import re
from datetime import datetime

###ABRINDO ARQUIVOS

In [2]:
with open("/content/drive/MyDrive/olist_orders_dataset.csv", "r", newline='', encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    pedidos = list(leitor)

In [3]:
with open("/content/drive/MyDrive/olist_products_dataset.csv", "r", newline='', encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    produtos = list(leitor)

###VALIDAÇÃO E TRATAMENTOS DE DADOS AUSENTES

In [4]:
def tratar_dados_ausentes(produtos):

    categorias_corrigidas = 0
    dimensoes_corrigidas = 0

    campos_dimensoes = [
        'product_weight_g',
        'product_length_cm',
        'product_height_cm',
        'product_width_cm'
    ]

    # ==========================
    # 1. Calcular médias
    # ==========================
    medias = {}

    for campo in campos_dimensoes:
        soma = 0
        quantidade = 0

        for produto in produtos:
            valor = produto[campo].strip()

            if valor != "":
                soma += float(valor)
                quantidade += 1

        medias[campo] = soma / quantidade if quantidade > 0 else 0

    # ==========================
    # 2. Corrigir dados ausentes
    # ==========================
    for produto in produtos:

        # Corrigir categoria
        if produto['product_category_name'].strip() == "":
            produto['product_category_name'] = "Sem Categoria"
            categorias_corrigidas += 1

        # Corrigir dimensões
        for campo in campos_dimensoes:

            if produto[campo].strip() == "":
                produto[campo] = str(round(medias[campo], 2))
                dimensoes_corrigidas += 1

    return (
        produtos,
        categorias_corrigidas,
        dimensoes_corrigidas
    )

###PADRONIZAÇÃO DE STRINGS E REGEX

In [5]:
import re

def padronizar_categorias(produtos):

    for produto in produtos:

        categoria = produto['product_category_name']

        categoria = categoria.lower().strip()

        categoria = re.sub(r'[^a-z0-9_ ]', '', categoria)

        produto['product_category_name'] = categoria

    return produtos

###LÓGICA DE REGRA DE NEGÓCIO (FILTROS E VALIDAÇÃO)

In [6]:
def validar_entregas_nulas(pedidos):

    entrega_nula_cancelado = 0
    entrega_nula_nao_cancelado = 0

    for pedido in pedidos:

        entrega = pedido[
            'order_delivered_customer_date'
        ]

        status = pedido['order_status']

        if entrega.strip() == "":

            if status == "canceled":
                entrega_nula_cancelado += 1

            else:
                entrega_nula_nao_cancelado += 1

    return (
        entrega_nula_cancelado,
        entrega_nula_nao_cancelado
    )

In [7]:
# Chamada da função
entrega_cancelado, entrega_nao_cancelado = validar_entregas_nulas(pedidos)

# Verificação da hipótese
if entrega_nao_cancelado == 0:
    print("Hipótese confirmada.")
else:
    print("Hipótese rejeitada.")

Hipótese rejeitada.


###FORTAMAÇÃO TEMPORAL (DATETIME)

In [8]:
from datetime import datetime

def converter_datas(pedidos):

    for pedido in pedidos:

        data = pedido['order_approved_at']

        if data.strip() != "":

            data_obj = datetime.strptime(
                data,
                "%Y-%m-%d %H:%M:%S"
            )

            pedido['order_approved_at'] = (
                data_obj.strftime("%d/%m/%Y")
            )

    return pedidos

###RELATÓRIO DE STATUS MANUAL

In [9]:
def gerar_relatorio(
        total_produtos,
        total_pedidos,
        categorias_corrigidas,
        dimensoes_corrigidas,
        cancelados):

    print("\n===== RELATÓRIO FINAL =====")

    print(
        f"Produtos processados: "
        f"{total_produtos}"
    )

    print(
        f"Pedidos processados: "
        f"{total_pedidos}"
    )

    print(
        f"Categorias corrigidas: "
        f"{categorias_corrigidas}"
    )

    print(
        f"Dimensões corrigidas: "
        f"{dimensoes_corrigidas}"
    )

    print(
        f"Pedidos cancelados: "
        f"{cancelados}"
    )

    print("\nBase sanitizada com sucesso.")

###MAIN

In [10]:
# FASE 1
produtos, categorias_corrigidas, dimensoes_corrigidas = (
    tratar_dados_ausentes(produtos)
)

# FASE 2
produtos = padronizar_categorias(produtos)

# FASE 3
entrega_cancelado, entrega_nao_cancelado = (
    validar_entregas_nulas(pedidos)
)

# FASE 4
pedidos = converter_datas(pedidos)

# Define total_cancelados before it's used in FASE 5
total_cancelados = sum(1 for pedido in pedidos if pedido['order_status'] == 'canceled')

# FASE 5
gerar_relatorio(
    len(produtos),
    len(pedidos),
    categorias_corrigidas,
    dimensoes_corrigidas,
    total_cancelados
)


===== RELATÓRIO FINAL =====
Produtos processados: 32951
Pedidos processados: 99441
Categorias corrigidas: 610
Dimensões corrigidas: 8
Pedidos cancelados: 625

Base sanitizada com sucesso.
